In [ ]:
!pip install datasets
from datasets import load_dataset

In [ ]:
# load data
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data # 这里显示数据集有train、validation、test三个划分

In [ ]:
data["train"][0, -1]

In [ ]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True, # 返回所有类别的分数
    device="cuda:0"
)

In [ ]:
output = pipe(
    data["test"][0]["text"],
    top_k=None
)

print(output)
# 这部分代码是debug用，排查单次输出的结果
# 也正是通过这部分代码的打印发现topk=None返回的顺序不是严格按照0是negative，2是positive来的，所以后续的计算出错了,是按照分数大小顺序来的

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset
## 创建一个惰性迭代器（Lazy Iterator），能够高效地将Pipeline应用到数据集的某个指定列上，而不会一次性将所有数据加载到内存中。

y_pred = []
# 注意这个地方目前模型默认topk=1，所以只返回一个值，是字典而非字典数组，所以指定topk=None符合当前的写法
for output in tqdm(pipe(KeyDataset(data["test"], "text"), top_k=None),
total=len(data["test"])):
  score_dict = {
        item["label"]: item["score"]
        for item in output
    }
  negative_score = score_dict["negative"]
  positive_score = score_dict["positive"]
  assignment = np.argmax(
        [negative_score, positive_score]
    )
  y_pred.append(assignment)
# 这里必须根据先根据标签分类再去比较分数，因为不是严格按照第一个是negative这个顺序返回的

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names=["Negative Review", "Positive Review"],
  )
  print(performance)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
!pip install sentence_transformers
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

In [ ]:
train_embeddings.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

## 当使用embedding模型做分类任务时，需要额外训练一个分类器，这里选择的是几率回归模型
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

In [ ]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

In [ ]:
# 假如是unseen label，将label稍加解释并同时embed
# 通过使label的描述更加具体和精确，能够提高最终结果的准确
label_embeddings = model.encode(["A very negative movie review", "A very positive movie review"])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim_matrix = cosine_similarity(test_embeddings, label_embeddings)

y_pred = np.argmax(sim_matrix, axis=1)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)